# 🎯 Attention-LSTM Training Notebook

This notebook loads historical price data from all asset classes, prepares time-series sequences, trains an Attention-based LSTM model, and saves it as `trained-attention-lstm.pth`.


## 📦 Block 1: Imports

In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

## 🧠 Block 2: Attention-Based LSTM Model

In [2]:
class AttentionLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, output_dim=1):
        super(AttentionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def attention(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        return self.fc(context)

## 📥 Block 3: Load Historical Data from All Asset Classes

In [3]:
def load_all_time_series(data_root='data/historical-data', seq_len=60):
    X, y = [], []
    for root, _, files in os.walk(data_root):
        for file in files:
            if file.endswith('.csv'):
                df = pd.read_csv(os.path.join(root, file))
                if 'Close' not in df.columns or len(df) < seq_len + 1:
                    continue
                series = df['Close'].values[-(seq_len + 1):]
                scaled = MinMaxScaler().fit_transform(series.reshape(-1, 1)).flatten()
                X.append(scaled[:-1])
                y.append(scaled[-1])
    X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)
    return X_tensor, y_tensor

## 🔁 Block 4: Train the Model

In [4]:
'''
✅ Why Use Tolerance-Based Accuracy?
It gives an intuitive sense of “how often the model was close enough,” which feels like classification-style accuracy — useful for business-facing decisions.
'''
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# Accuracy metric: % of predictions within ±10% of actual value
def percentage_accuracy(y_true, y_pred, tolerance=0.10):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    within_tolerance = np.abs(y_true - y_pred) <= (tolerance * np.abs(y_true))
    return np.mean(within_tolerance) * 100

X, y = load_all_time_series(data_root='./data/historical-data')
print("Training samples:", X.shape[0])

dataset = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

model = AttentionLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

final_accuracy = 0

for epoch in range(100):
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch_X, batch_y in dataset:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = loss_fn(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        all_preds.extend(output.detach().cpu().numpy())
        all_targets.extend(batch_y.detach().cpu().numpy())

    # Print loss each epoch
    print(f"Epoch {epoch:02d} - MSE Loss: {total_loss:.6f}")

    # Save final accuracy at last epoch
    if epoch == 49:
        final_accuracy = percentage_accuracy(all_targets, all_preds)

# ✅ Final accuracy after training
print(f"\n Accuracy: {final_accuracy:.2f}%")


/var/folders/33/tmt_dn957d99k_t2zvbqlw_40000gn/T/ipykernel_26609/2580702717.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)


Training samples: 546
Epoch 00 - MSE Loss: 5.102226
Epoch 01 - MSE Loss: 1.413275
Epoch 02 - MSE Loss: 1.252622
Epoch 03 - MSE Loss: 1.190133
Epoch 04 - MSE Loss: 1.411365
Epoch 05 - MSE Loss: 1.189832
Epoch 06 - MSE Loss: 1.198432
Epoch 07 - MSE Loss: 1.186980
Epoch 08 - MSE Loss: 1.163911
Epoch 09 - MSE Loss: 1.177982
Epoch 10 - MSE Loss: 1.198819
Epoch 11 - MSE Loss: 1.418601
Epoch 12 - MSE Loss: 1.416307
Epoch 13 - MSE Loss: 1.323823
Epoch 14 - MSE Loss: 1.157527
Epoch 15 - MSE Loss: 1.201817
Epoch 16 - MSE Loss: 1.184694
Epoch 17 - MSE Loss: 1.205644
Epoch 18 - MSE Loss: 1.163479
Epoch 19 - MSE Loss: 1.131199
Epoch 20 - MSE Loss: 1.361849
Epoch 21 - MSE Loss: 1.176393
Epoch 22 - MSE Loss: 1.230177
Epoch 23 - MSE Loss: 1.167703
Epoch 24 - MSE Loss: 1.161796
Epoch 25 - MSE Loss: 1.276870
Epoch 26 - MSE Loss: 1.068542
Epoch 27 - MSE Loss: 1.081436
Epoch 28 - MSE Loss: 1.215574
Epoch 29 - MSE Loss: 1.191864
Epoch 30 - MSE Loss: 1.082069
Epoch 31 - MSE Loss: 1.027320
Epoch 32 - MSE Los

## 💾 Block 5: Save the Trained Model

In [5]:
torch.save(model.state_dict(), 'trained-attention-lstm.pth')
print("✅ Model saved as trained-attention-lstm.pth")

✅ Model saved as trained-attention-lstm.pth
